# 02 — Store Segmentation and Assortment Opportunity
## Iowa Liquor Sales, January–July 2026

**Business question:** Which stores should a distributor prioritize, and which product categories appear underrepresented relative to similar stores?

This notebook uses the cleaned table from Notebook 01 to build interpretable store segments and a peer-category benchmark. The benchmark gap is a screening signal, **not a forecast of incremental revenue**.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 50)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "iowa_liquor_sales_2026_clean.parquet"
if not PROCESSED_PATH.exists():
    raise FileNotFoundError("Run notebooks/01_data_audit_cleaning.ipynb first.")
df = pd.read_parquet(PROCESSED_PATH)
assert len(df) == 1_402_639
assert df["ordered_on"].min() == pd.Timestamp("2026-01-01")
assert df["ordered_on"].max() == pd.Timestamp("2026-07-31")
print(f"Rows: {len(df):,} | Stores: {df['store_no'].nunique():,}")


## 1. Build store-level features

The clustering features are intentionally business-readable: **gross sales, ordering days, SKU breadth, and category breadth**. Returns, invoice value, and active months are retained for profiling but do not define the clusters.


In [ ]:
df["gross_sales_line"] = df["sales_dollars"].clip(lower=0)
df["return_dollars"] = -df["sales_dollars"].clip(upper=0)
df["month"] = df["ordered_on"].dt.to_period("M").astype(str)
latest = (df.sort_values("ordered_on").groupby("store_no").agg(
    store_name=("store_name", "last"), city=("store_city", "last"), county=("county_name", "last")
).reset_index())
store = (df.groupby("store_no").agg(
    net_sales=("sales_dollars", "sum"), gross_sales=("gross_sales_line", "sum"),
    return_dollars=("return_dollars", "sum"), invoices=("invoice_id", "nunique"),
    order_days=("ordered_on", "nunique"), unique_skus=("item_no", "nunique"),
    unique_categories=("category_code", "nunique"), active_months=("month", "nunique")
).reset_index().merge(latest, on="store_no", how="left"))
store["avg_invoice_value"] = store["net_sales"] / store["invoices"]
store["return_rate"] = np.where(store["gross_sales"] > 0, store["return_dollars"] / store["gross_sales"], np.nan)
store.describe(percentiles=[.25,.5,.75,.9,.95]).T


## 2. Select and fit the store segments

The four clustering features are right-skewed, so they are transformed with `log1p` and standardized. Silhouette scores for the verified snapshot are approximately **0.446, 0.442, 0.369, 0.355, 0.321** for k=2 through k=6. We choose **k=3** because it is nearly as well separated as k=2 but yields a more useful Developing / Core / Strategic account structure.


In [ ]:
FEATURES = ["gross_sales", "order_days", "unique_skus", "unique_categories"]
X_scaled = StandardScaler().fit_transform(np.log1p(store[FEATURES]))
scores, models = {}, {}
for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = model.fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)
    models[k] = model
print(pd.Series(scores, name="silhouette_score"))

store["cluster"] = models[3].labels_
order = store.groupby("cluster")["gross_sales"].median().sort_values().index.tolist()
store["segment"] = store["cluster"].map({order[0]: "Developing", order[1]: "Core", order[2]: "Strategic"})
summary = store.groupby("segment").agg(
    stores=("store_no", "count"), gross_sales=("gross_sales", "sum"),
    median_gross_sales=("gross_sales", "median"), median_order_days=("order_days", "median"),
    median_unique_skus=("unique_skus", "median"), median_unique_categories=("unique_categories", "median")
)
summary["sales_share"] = summary["gross_sales"] / summary["gross_sales"].sum()
summary = summary.loc[["Developing", "Core", "Strategic"]]
summary


### Verified segment profile

| Segment | Stores | Median gross sales | Median order days | Median SKUs | Median categories | Share of gross sales |
|---|---:|---:|---:|---:|---:|---:|
| Developing | 274 | $7,497 | 4 | 38 | 13 | 1.1% |
| Core | 1,087 | $29,881 | 17 | 95 | 21 | 15.2% |
| Strategic | 822 | $116,568 | 31 | 321 | 36 | 83.8% |

The Strategic tier is about 38% of stores but contributes roughly **84% of gross sales**, making it the first account tier to review.


In [ ]:
plt.figure(figsize=(9, 6))
for segment in ["Developing", "Core", "Strategic"]:
    part = store[store["segment"] == segment]
    plt.scatter(part["unique_skus"], part["gross_sales"], alpha=.55, s=20, label=segment)
plt.xscale("log"); plt.yscale("log")
plt.xlabel("Unique SKUs purchased (log scale)"); plt.ylabel("Gross sales ($, log scale)")
plt.title("Store segments by scale and assortment breadth"); plt.legend(); plt.show()


## 3. Build peer-category benchmarks

Within each segment, a category qualifies only if it represents at least **1% of segment gross sales** and is purchased by at least **50% of stores**. This avoids turning niche categories into false opportunities.


In [ ]:
df["segment"] = df["store_no"].map(store.set_index("store_no")["segment"])
store_category = (df.groupby(["store_no", "category_code", "category_name"], dropna=False)["gross_sales_line"]
                  .sum().rename("category_sales").reset_index()
                  .merge(store[["store_no","store_name","city","county","segment","gross_sales","active_months"]], on="store_no", how="left"))
segment_category = (store_category.groupby(["segment","category_code","category_name"])
                    .agg(segment_category_sales=("category_sales","sum"), stores_carrying=("store_no","nunique")).reset_index())
segment_totals = store.groupby("segment").agg(segment_sales=("gross_sales","sum"), segment_stores=("store_no","count")).reset_index()
segment_category = segment_category.merge(segment_totals, on="segment")
segment_category["benchmark_share"] = segment_category["segment_category_sales"] / segment_category["segment_sales"]
segment_category["penetration"] = segment_category["stores_carrying"] / segment_category["segment_stores"]
bench = segment_category[(segment_category["benchmark_share"] >= .01) & (segment_category["penetration"] >= .50)].copy()
bench.groupby("segment").size().rename("qualified_categories")


In the verified snapshot, **24 categories** qualify for Strategic stores and **16** for Core stores. The largest Strategic category shares include American vodkas, Canadian whiskies, straight bourbon whiskies, 100% agave tequila, and spiced rum.


## 4. Screen for under-indexed assortment gaps

The first action list focuses on Core and Strategic stores active in at least five months. A store-category pair is flagged when actual category sales are below **50% of its peer-mix benchmark** and the seven-month benchmark gap is at least **$5,000**.


In [ ]:
eligible = store[store["segment"].isin(["Core","Strategic"]) & store["active_months"].ge(5)].copy()
parts = []
for segment in ["Core", "Strategic"]:
    s = eligible[eligible["segment"] == segment][["store_no","store_name","city","county","gross_sales","segment"]]
    c = bench[bench["segment"] == segment][["category_code","category_name","benchmark_share","penetration"]]
    parts.append(s.assign(_key=1).merge(c.assign(_key=1), on="_key").drop(columns="_key"))
opportunities = pd.concat(parts, ignore_index=True)
actual = store_category[["store_no","category_code","category_sales"]]
opportunities = opportunities.merge(actual, on=["store_no","category_code"], how="left")
opportunities["category_sales"] = opportunities["category_sales"].fillna(0)
opportunities["expected_at_peer_mix"] = opportunities["gross_sales"] * opportunities["benchmark_share"]
opportunities["benchmark_gap"] = (opportunities["expected_at_peer_mix"] - opportunities["category_sales"]).clip(lower=0)
opportunities["share_index"] = opportunities["category_sales"] / opportunities["expected_at_peer_mix"]
candidates = opportunities[(opportunities["share_index"] < .50) & (opportunities["benchmark_gap"] >= 5000)].sort_values("benchmark_gap", ascending=False)
print(f"Candidate store-category gaps: {len(candidates):,}")
print(f"Stores represented: {candidates['store_no'].nunique():,}")
candidates[["store_no","store_name","segment","gross_sales","category_name","category_sales","expected_at_peer_mix","benchmark_gap","share_index"]].head(20)


## Business interpretation and limitations

1. **Strategic accounts first:** 822 stores drive about 83.8% of gross sales.
2. **Core stores are the growth pipeline:** they contribute another 15.2% and carry materially narrower assortments.
3. **A gap is a review trigger, not an automatic stocking recommendation.** Store format, chain strategy, local demand, and shelf constraints are not modeled.
4. **Do not interpret benchmark gap as incremental revenue.** It is the difference between actual category mix and peer category mix.
5. The analysis covers only January–July 2026, and precise margin analysis remains constrained by the January source-precision issue documented in Notebook 01.

A strong next extension would add chain/format-aware peer groups and move from category gaps to specific SKU recommendations and invoice-level product affinity.


## Portfolio takeaway

**raw public data → quality audit → reproducible cleaning → feature engineering → model selection → interpretable segmentation → peer benchmarking → prioritized action list → limitations**

The clustering is not the end product. Its purpose is to create a defensible peer structure for a business decision.
